---
# Методы статистической обработки информации
---
## Лабораторная работа 2 / Регрессионный анализ / Вариант 15
---

---
## Подготовка
---

### Импорт библиотек

In [243]:
from math             import log10
from MoSDP_data       import get_data
from MoSDP_operations import *

### Получение данных варианта

In [244]:
variant  = 15 # int(input("Введите номер варианта: "))
var_data = get_data(variant)

### Подготовка данных


In [245]:
Y = transpose([var_data[0]])
X = transpose(var_data[1:])

n = len(X)
k = len(X[0])

for x in X:
    x.insert(0, 1)

---
## Задание 1
---

### Функция "Оценка уравнения регрессии"

In [246]:
def estimate(Y, X):
    xTx = matrix_multiplication(transpose(X), X)
    xTy = matrix_multiplication(transpose(X), Y)
    b   = matrix_multiplication(inverse_matrix(xTx), xTy)

    return b

### Функция "Формирование линейного уравнения"

In [247]:
def draw_linear(b):
    equation = f"y = {b[0][0]}"

    for i in range(1, len(b)):
        if b[i][0] > 0:
            equation += f" + {b[i][0]}"
        else:
            equation += f" - {abs(b[i][0])}"

        equation += f" * x{i}"

    return equation

### Оценка линейного уравнения регрессии

In [248]:
b_linear = estimate(Y, X)
print("b_linear:", b_linear, "\n")

print(draw_linear(b_linear))

b_linear: [[-12.560998303339147], [18.21127665232234], [1.6273983943464145], [0.5084514879716515], [0.2785008638589499], [0.21507667359831828]] 

y = -12.560998303339147 + 18.21127665232234 * x1 + 1.6273983943464145 * x2 + 0.5084514879716515 * x3 + 0.2785008638589499 * x4 + 0.21507667359831828 * x5


### Функция "Формирование степенного уравнения"

In [249]:
def draw_pow(b):
    equation = f"y = {b[0][0]}"

    for i in range(1, len(b)):
        equation += f" * x{i}^({b[i][0]})"

    return equation

### Оценка степенного уравнения регрессии

In [250]:
Y_lg = copy(Y)
X_lg = copy(X)

for i in range(len(X)):
    Y_lg[i][0] = log10(Y_lg[i][0])

    for j in range(1, len(X[0])):
        X_lg[i][j] = log10(X_lg[i][j])

b_pow       = estimate(Y_lg, X_lg)
b_pow[0][0] = 10 ** b_pow[0][0]

print("b_exp:", b_pow, "\n")
print(draw_pow(b_pow))

b_exp: [[0.5749714020371827], [3.591015023519958], [-0.027089077716006216], [0.24718836073139272], [0.2095603646767561], [1.0877541678012221]] 

y = 0.5749714020371827 * x1^(3.591015023519958) * x2^(-0.027089077716006216) * x3^(0.24718836073139272) * x4^(0.2095603646767561) * x5^(1.0877541678012221)


---
## Задание 2
---

### Функция "Значимость уравнения"

In [251]:
def significance_equation(Y, X, b):
    Y_reg = matrix_multiplication(X, b)

    Q = 0
    for i in range(len(Y)):
        Q += (Y[i][0] - Y_reg[i][0]) ** 2

    S_2 = Q / (n - k - 1)

    Q_r = matrix_multiplication(transpose(Y_reg), Y_reg)

    F_critical = 2.25 # По "таблице 4": a = 0.05; v1 = k + 1 = 5 + 1 = 6 и v2 = n - k - 1 = 53 - 5 - 1 = 47
    F          = (Q_r[0][0] / 2) / S_2

    res = ('<=', 'не ')
    if F > F_critical:
        res = ('>', '')

    print(f"F_набл = {F} {res[0]} {F_critical}: {res[1]}значимое")

    return S_2

### Значимость линейного уравнения

In [258]:
S_2_linear = significance_equation(Y, X, b_linear)

F_набл = 251.224052719012 > 2.25: значимое


### Значимость степенного уравнения

In [253]:
b_pow[0][0] = log10(b_pow[0][0])

S_2_pow = significance_equation(Y_lg, X_lg, b_pow)

F_набл = 1303.8852814673899 > 2.25: значимое


### Функция "Значимость коэффициентов регрессии"

In [254]:
def significance_coefficients(X, b, S_2):
    xTx = inverse_matrix(matrix_multiplication(transpose(X), X))
    S   = matrix_by_a_number(xTx, S_2)

    t_critical = 2.021 # По "таблице 2": a = 0.05 и v = n - k - 1 = 53 - 5 - 1 = 47

    for i in range(1, len(S)):
        t = b[i][0] / (S[i][i] ** 0.5)

        res = (' <=', 'не')
        if t > t_critical:
            res = (' > ', '  ')

        print(f"t_набл(b{i}) = {t:{7}.3f} {res[0]} {t_critical}: {res[1]} значимый")

### Значимость коэффициентов регрессии линейного уравнения

In [255]:
significance_coefficients(X, b_linear, S_2_linear)

t_набл(b1) =   0.860  <= 2.021: не значимый
t_набл(b2) =   0.598  <= 2.021: не значимый
t_набл(b3) =   0.428  <= 2.021: не значимый
t_набл(b4) =   1.457  <= 2.021: не значимый
t_набл(b5) =   0.906  <= 2.021: не значимый


### Значимость коэффициентов регрессии степенного уравнения

In [256]:
significance_coefficients(X_lg, b_pow, S_2_pow)

t_набл(b1) =   2.524  >  2.021:    значимый
t_набл(b2) =  -0.438  <= 2.021: не значимый
t_набл(b3) =   1.420  <= 2.021: не значимый
t_набл(b4) =   1.818  <= 2.021: не значимый
t_набл(b5) =   2.571  >  2.021:    значимый


---
## Задание 3
---

### ...

In [257]:
...

Ellipsis